# Notebook for plotting a clustering
Needs to be run after `..._clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path(os.getcwd()), *Path(os.getcwd()).parents] if path.stem == "esa_consulting")
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
import plotnine as p9

from utils.misc_utils import reduce_dimen_2d
from utils.plotting_utils import make_plot_2d_data
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_imputed = DatasetBundle.load(Path("data/imputed/data_discov_imputed"))
data_discov_imputed.meta = cast(pd.DataFrame, data_discov_imputed.meta)
data_discov_imputed.na = cast(pd.DataFrame, data_discov_imputed.na)
data_valid_imputed = DatasetBundle.load(Path("data/imputed/data_valid_imputed"))
data_valid_imputed.meta = cast(pd.DataFrame, data_valid_imputed.meta)
data_valid_imputed.na = cast(pd.DataFrame, data_valid_imputed.na)

cluster_pred_discov = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_discov.parquet"))
cluster_pred_valid = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_valid.parquet"))

# Set outputs folder
outputs_folder = "outputs/gmm_clustering"

In [ ]:
# Plot projected clustering in 2d
projection_method = "TSNE"  # "UMAP"
tsne_kwargs = dict(perplexity=100, early_exaggeration=20)

# Reduce data dimensionality using the specified projection method (time intensive)
data_discov_2d = reduce_dimen_2d(
    data_discov_imputed.data, projection_method=projection_method, rng=np.random.default_rng(23523), **tsne_kwargs
)
data_valid_2d = reduce_dimen_2d(
    data_valid_imputed.data, projection_method=projection_method, rng=np.random.default_rng(4086), **tsne_kwargs
)

# Plot clusterings
clusters_discov_plot = make_plot_2d_data(data=data_discov_2d, cluster_labels=cluster_pred_discov["label"])
clusters_valid_plot = make_plot_2d_data(data=data_valid_2d, cluster_labels=cluster_pred_valid["label"])

# Combine the three plots
clusters_discov_plot += p9.labs(title="Discovery clustering")
clusters_valid_plot += p9.labs(title="Validation clustering")
clusters_all_plot = (clusters_discov_plot + clusters_valid_plot) & p9.theme(figure_size=(10, 5))

clusters_all_plot.show()

# Save plots
# save_plot(clusters_discov_plot, Path(outputs_folder, f"clusters_discov_plot_{projection_method.lower()}.png"), dpi=150)
# save_plot(clusters_valid_plot, Path(outputs_folder, f"clusters_valid_plot_{projection_method.lower()}.png"), dpi=150)

save_plot(clusters_all_plot, Path(outputs_folder, f"clusters_all_plot_{projection_method.lower()}.png"), dpi=150)